# Fe3O4 - Curvas de Histerese (BR1, BR2, BR3, COM2)

Notebook único juntando o tratamento dos dados do SQUID e os gráficos MxH das quatro amostras.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

## Parâmetros das amostras

Massas (g) e caminhos dos arquivos `.dat` exportados do SQUID.

In [ ]:
# massas medidas na balança (g)
massa1_g = 0.0481   # BR1
massa2_g = 0.0777   # BR2
massa3_g = 0.0250   # BR3
massa_COM2 = 0.0284 # COM2

# ajustar os caminhos conforme a pasta local
pasta = r"C:\Users\Wesll\OneDrive\Documents\Artigo\Artigo_Argila_ferro"

arquivo_BR1_300K = pasta + r"\BR1_MxH_300K_NEW.dat"
arquivo_BR1_50K  = pasta + r"\BR1_MxH_50K_new.dat"

arquivo_BR2_300K = pasta + r"\BR2_MxH_300K.dat"
arquivo_BR2_50K  = pasta + r"\BR2_MxH_50K_new.dat"

arquivo_BR3_300K = pasta + r"\BR3_MxH_300K.dat"
arquivo_BR3_50K  = pasta + r"\BR3_MxH_50K_new.dat"

# COM2 veio num arquivo só, com as duas temperaturas misturadas
arquivo_COM2 = pasta + r"\COM_2_MxH_300K.dat"

coluna_campo = 'Magnetic Field (Oe)'
coluna_momento = 'DC Moment Fixed Ctr (emu)'

## Leitura dos dados do SQUID

Os arquivos `.dat` têm um cabeçalho longo antes de `[Data]`; a função abaixo pula esse cabeçalho e devolve o DataFrame.

In [ ]:
def ler_dados_squid(caminho_arquivo):
    # acha a linha onde o cabeçalho termina e os dados começam
    with open(caminho_arquivo, 'r', encoding='latin-1') as f:
        for i, linha in enumerate(f):
            if '[Data]' in linha:
                linha_inicio = i + 1
                break

    df = pd.read_csv(caminho_arquivo, skiprows=linha_inicio, encoding='latin-1', on_bad_lines='skip')
    return df

## Leitura e normalização pela massa

Magnetização (emu/g) = Momento (emu) / massa (g), para cada amostra.

In [ ]:
# BR1
df_BR1_300K = ler_dados_squid(arquivo_BR1_300K)
df_BR1_50K = ler_dados_squid(arquivo_BR1_50K)

campo_BR1_300K = df_BR1_300K[coluna_campo].values
mag_BR1_300K = (df_BR1_300K[coluna_momento] / massa1_g).values

campo_BR1_50K = df_BR1_50K[coluna_campo].values
mag_BR1_50K = (df_BR1_50K[coluna_momento] / massa1_g).values

In [ ]:
# BR2
df_BR2_300K = ler_dados_squid(arquivo_BR2_300K)
df_BR2_50K = ler_dados_squid(arquivo_BR2_50K)

campo_BR2_300K = df_BR2_300K[coluna_campo].values
mag_BR2_300K = (df_BR2_300K[coluna_momento] / massa2_g).values

campo_BR2_50K = df_BR2_50K[coluna_campo].values
mag_BR2_50K = (df_BR2_50K[coluna_momento] / massa2_g).values

In [ ]:
# BR3
df_BR3_300K = ler_dados_squid(arquivo_BR3_300K)
df_BR3_50K = ler_dados_squid(arquivo_BR3_50K)

campo_BR3_300K = df_BR3_300K[coluna_campo].values
mag_BR3_300K = (df_BR3_300K[coluna_momento] / massa3_g).values

campo_BR3_50K = df_BR3_50K[coluna_campo].values
mag_BR3_50K = (df_BR3_50K[coluna_momento] / massa3_g).values

In [ ]:
# COM2 - aqui as duas temperaturas estão no mesmo arquivo, separo pela coluna de temperatura
df_COM2 = ler_dados_squid(arquivo_COM2)
col_temp = [col for col in df_COM2.columns if 'Temp' in col][0]

df_COM2_300K = df_COM2[np.isclose(df_COM2[col_temp], 300, atol=5)]
df_COM2_50K = df_COM2[np.isclose(df_COM2[col_temp], 50, atol=5)]

campo_COM2_300K = df_COM2_300K[coluna_campo].values
mag_COM2_300K = (df_COM2_300K[coluna_momento] / massa_COM2).values

campo_COM2_50K = df_COM2_50K[coluna_campo].values
mag_COM2_50K = (df_COM2_50K[coluna_momento] / massa_COM2).values

## Função de plotagem

Mesmo padrão usado nos gráficos do artigo: curva principal + inset com a região de baixo campo. Recebo o `ax` de fora para poder reusar tanto nos gráficos individuais quanto no painel final.

In [ ]:
def plot_histerese(ax, campo_300K, mag_300K, campo_50K, mag_50K, titulo, limite_inset=500):
    ax.axhline(0, color='black', linewidth=0.8, zorder=1)
    ax.axvline(0, color='black', linewidth=0.8, zorder=1)
    ax.tick_params(axis='both', which='major', labelsize=11)
    ax.grid(True, which='major', linestyle=':', color='gray', alpha=0.4, zorder=0)

    ax.plot(campo_300K / 1000, mag_300K, color='red', linewidth=1.5, label='300 K', alpha=0.9, zorder=3)
    ax.plot(campo_50K / 1000, mag_50K, color='#1f77b4', linewidth=1.5, label='50 K', alpha=0.9, zorder=3)

    ax.set_xlabel('Magnetic Field (kOe)', fontsize=13)
    ax.set_ylabel('Magnetization (emu/g)', fontsize=13)
    ax.set_title(titulo, fontsize=14, fontweight='bold', pad=12)
    ax.legend(fontsize=10, frameon=True, shadow=True, loc='upper left')

    # inset com a região de baixo campo
    axins = inset_axes(ax, width="35%", height="35%", loc='lower right', borderpad=3)

    mask_300K = (campo_300K >= -limite_inset) & (campo_300K <= limite_inset)
    mask_50K = (campo_50K >= -limite_inset) & (campo_50K <= limite_inset)

    axins.set_title('Low Field Region', fontsize=10, fontweight='bold', pad=6)
    axins.plot(campo_300K[mask_300K], mag_300K[mask_300K], '-o', color='red', linewidth=1, markersize=3, alpha=0.8)
    axins.plot(campo_50K[mask_50K], mag_50K[mask_50K], '-s', color='#1f77b4', linewidth=1, markersize=3, alpha=0.8)

    axins.set_xlim(-limite_inset, limite_inset)
    y_min = min(mag_300K[mask_300K].min(), mag_50K[mask_50K].min())
    y_max = max(mag_300K[mask_300K].max(), mag_50K[mask_50K].max())
    axins.set_ylim(y_min * 1.1, y_max * 1.1)

    axins.set_xlabel('Magnetic Field (Oe)', fontsize=9)
    axins.set_ylabel('Magnetization (emu/g)', fontsize=9)
    axins.grid(False)
    axins.axhline(0, color='black', linewidth=0.5, alpha=0.5)
    axins.axvline(0, color='black', linewidth=0.5, alpha=0.5)
    axins.tick_params(axis='both', which='major', labelsize=8)

    return ax

## Tabela resumo (Ms, Hc, Mr)

Valores tirados das curvas de cada amostra, já usados no artigo.

In [ ]:
colunas = ["Sample", "Ms 300K (emu/g)", "Hc 300K (Oe)", "Mr 300K (emu/g)",
           "Ms 50K (emu/g)", "Hc 50K (Oe)", "Mr 50K (emu/g)"]

dados = [
    ["BR1", 17.32, 24, 1.22, 21.37, 54, 1.46],
    ["BR2", 6.45, 30, 0.61, 8.40, 131, 1.85],
    ["BR3", 4.36, 45, 0.50, 6.11, 131, 0.70],
    ["COM2", 15.74, 25, 1.83, 19.57, 48, 1.52],
]

fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')
tabela = ax.table(cellText=dados, colLabels=colunas, loc='center', cellLoc='center')
tabela.auto_set_font_size(False)
tabela.set_fontsize(12)
tabela.scale(1, 1.8)

for (row, col), cell in tabela.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold')
        cell.set_facecolor('#f2f2f2')
        cell.set_linewidth(1.2)
    else:
        cell.set_linewidth(0.8)

plt.tight_layout()
plt.show()

## Gráficos finais

As quatro curvas juntas num painel único, pronto pro artigo.

In [ ]:
plt.style.use('seaborn-v0_8-ticks')

fig, axs = plt.subplots(2, 2, figsize=(14, 11))

plot_histerese(axs[0, 0], campo_BR1_300K, mag_BR1_300K, campo_BR1_50K, mag_BR1_50K, 'Fe$_3$O$_4$ (BR1)')
plot_histerese(axs[0, 1], campo_BR2_300K, mag_BR2_300K, campo_BR2_50K, mag_BR2_50K, 'Fe$_3$O$_4$ (BR2)')
plot_histerese(axs[1, 0], campo_BR3_300K, mag_BR3_300K, campo_BR3_50K, mag_BR3_50K, 'Fe$_3$O$_4$ (BR3)')
plot_histerese(axs[1, 1], campo_COM2_300K, mag_COM2_300K, campo_COM2_50K, mag_COM2_50K, 'Fe$_3$O$_4$ (COM2)')

letras = ['(a)', '(b)', '(c)', '(d)']
for ax, letra in zip(axs.flatten(), letras):
    ax.text(0.02, 0.97, letra, transform=ax.transAxes, fontsize=13, fontweight='bold', va='top')

plt.tight_layout()

# salvar direto na pasta do artigo
nome_arquivo = 'Histerese_Fe3O4_todas_amostras.png'
plt.savefig(f'{pasta}\\{nome_arquivo}', dpi=300, bbox_inches='tight')
print(f"Gráfico salvo em: {pasta}")

plt.show()